# Columns:
- time, device, active_power, wind_speed, air_density, wind_direction,
- nacelle_direction, nacelle_position, ambient_temp, rotor_speed,
- generator_speed, gearbox_oil_temp, generator_temp, bearing_temp,
- converter_temp, pitch_blade_angle_1, pitch_blade_angle_2,
- pitch_blade_angle_3, is_drifted, fault_type, drift_start_time,
- fault_severity

# Devices:
- WT01_healthy: no injected fault
- WT02_temp_drift: gradual temperature drift
- WT03_pitch_drift: abrupt pitch misalignment
- WT04_yaw_drift: gradual yaw misalignment
- WT05_temp_drift_recovers: gradual temperature drift, gradual recovery
- WT06_pitch_drift_repaired: abrupt pitch misalignment, abrupt recovery
- WT07_yaw_drift_recovers: gradual yaw misalignment, gradual recovery
- WT08_temp_drift_repaired: gradual temperature drift, abrupt repair
- WT09_yaw_drift_repaired: gradual yaw misalignment, abrupt repair
- WT10_temp_then_yaw: gradual temperature drift, then gradual yaw misalignment, no repair

# INFO:
- 10 minute intervals
- active_power follows the supplied power curve
- air_density affects power
- yaw misalignment reduces power via cos(yaw_error)^3
- pitch misalignment makes blade angles diverge and reduces efficiency
- temperatures depend on ambient conditions and loading
- is_drifted = 1 when the injected fault is active, including ramp-up

# Note, when making models, avoid very similar signals.
#### In terms of wind turbine, its better to have a combined blade pitch signal than putting in all 3 blade pitch angles unless your model specifically tracks pitch misalignments etc.
#### Similarly for power level calculated signals. Can check the EMM VB repo for more examples. This repo is to help me generate data for my other research projects. And for future research.

In [22]:
import numpy as np
import pandas as pd
from pathlib import Path

In [23]:
random_seed = 42 # answer to all things in the universe xD

In [24]:
POWER_CURVE = [
    (0, 0), (1, 0), (2, 0), (3, 0),
    (3.5, 8), (4, 35), (4.5, 77), (5, 137), (5.5, 217),
    (6, 321), (6.5, 450), (7, 607), (7.5, 793), (8, 1008),
    (8.5, 1250), (9, 1515), (9.5, 1685), (10, 1790),
    (10.5, 1790), (11, 1790), (11.5, 1790), (12, 1790),
    (12.5, 1790), (13, 1790), (13.5, 1790), (14, 1790),
    (14.5, 1790), (15, 1790), (15.5, 1790), (16, 1790),
    (16.5, 1790), (17, 1790), (17.5, 1790), (18, 1790),
    (18.5, 1790), (19, 1790), (19.5, 1790), (20, 1790),
    (20.5, 1790), (21, 1790), (21.5, 1790), (22, 1790),
    (22.5, 1790), (23, 0), (23.5, 0), (24, 0), (24.5, 0),
    (25, 0), (25.5, 0), (26, 0), (26.5, 0), (27, 0),
    (27.5, 0), (28, 0), (28.5, 0), (29, 0), (29.5, 0), (30, 0),
]

WS_POINTS = np.array([x for x, _ in POWER_CURVE], dtype=float)
P_POINTS = np.array([y for _, y in POWER_CURVE], dtype=float)

OUTPUT_COLUMNS = [
    "time", "device", "active_power", "wind_speed", "air_density",
    "wind_direction", "nacelle_direction", "nacelle_position",
    "ambient_temp", "rotor_speed", "generator_speed",
    "gearbox_oil_temp", "generator_temp", "bearing_temp",
    "converter_temp", "pitch_blade_angle_1", "pitch_blade_angle_2",
    "pitch_blade_angle_3", "is_drifted", "fault_labels",
    "fault_severity", "drift_start_time"
]

FAULT_BINARY_COLUMNS = [
    "fault_temperature",
    "fault_pitch_misalignment",
    "fault_yaw_misalignment",
]

ALL_COLUMNS = OUTPUT_COLUMNS + FAULT_BINARY_COLUMNS

In [25]:
def wrap_deg(angle):
    return np.mod(angle, 360.0)


def angle_diff(target, source):
    return (target - source + 180.0) % 360.0 - 180.0


def power_curve_kw(wind_speed):
    return np.interp(wind_speed, WS_POINTS, P_POINTS)


def commanded_pitch_angle(wind_speed):
    ws = np.asarray(wind_speed, dtype=float)
    pitch = np.zeros_like(ws)

    pitch = np.where(ws < 3.0, 89.0, pitch)
    pitch = np.where((ws >= 3.0) & (ws < 10.0), 0.5 + 0.35 * (ws - 3.0), pitch)
    pitch = np.where((ws >= 10.0) & (ws < 15.0), 3.0 + 1.6 * (ws - 10.0), pitch)
    pitch = np.where((ws >= 15.0) & (ws < 20.0), 11.0 + 2.6 * (ws - 15.0), pitch)
    pitch = np.where((ws >= 20.0) & (ws < 23.0), 24.0 + 20.0 * (ws - 20.0), pitch)
    pitch = np.where(ws >= 23.0, 89.0, pitch)

    return np.clip(pitch, 0.0, 90.0)


def severity_label(values):
    labels = np.full(len(values), "none", dtype=object)
    labels[(values > 0.0) & (values < 0.33)] = "low"
    labels[(values >= 0.33) & (values < 0.66)] = "medium"
    labels[values >= 0.66] = "high"
    return labels



In [26]:
def build_fault_profile(n, start_idx, ramp_steps=1, end_idx=None, shape="linear", max_severity=1.0, ramp_down_steps=0):
    profile = np.zeros(n, dtype=float)

    if start_idx is None or start_idx >= n:
        return profile

    if end_idx is None:
        end_idx = n

    start_idx = max(0, start_idx)
    end_idx = min(end_idx, n)

    if end_idx <= start_idx:
        return profile

    if shape == "abrupt":
        profile[start_idx:end_idx] = max_severity
        return profile

    if shape == "linear":
        ramp_steps = max(1, int(ramp_steps))
        ramp_end = min(start_idx + ramp_steps, end_idx)
        if ramp_end > start_idx:
            profile[start_idx:ramp_end] = np.linspace(
                0.0, max_severity, ramp_end - start_idx, endpoint=False
            )
        profile[ramp_end:end_idx] = max_severity

        if ramp_down_steps:
            ramp_down_steps = int(ramp_down_steps)
            ramp_down_start = max(ramp_end, end_idx - ramp_down_steps)
            if end_idx > ramp_down_start:
                profile[ramp_down_start:end_idx] = np.linspace(
                    max_severity, 0.0, end_idx - ramp_down_start, endpoint=False
                )
        return profile

    if shape == "intermittent":
        ramp_steps = max(1, int(ramp_steps))
        ramp_end = min(start_idx + ramp_steps, end_idx)
        if ramp_end > start_idx:
            profile[start_idx:ramp_end] = np.linspace(
                0.0, max_severity, ramp_end - start_idx, endpoint=False
            )
        t = np.arange(ramp_end, end_idx)
        pattern = 0.5 * (1.0 + np.sin(2 * np.pi * (t - ramp_end) / (6 * 24 * 3)))
        profile[ramp_end:end_idx] = max_severity * (0.35 + 0.65 * pattern)
        return profile

    raise ValueError(f"Unsupported profile shape: {shape}")

In [27]:
def start_day_to_idx(start_day, steps_per_day):
    if start_day is None:
        return None
    return int(start_day * steps_per_day)


def make_device_params(rng, overrides=None):
    params = {
        "rated_power_kw": float(np.clip(rng.normal(1790.0, 15.0), 1700.0, 1850.0)),
        "gearbox_ratio": float(np.clip(rng.normal(95.0, 1.2), 90.0, 100.0)),
        "yaw_response_gain": float(np.clip(rng.normal(0.35, 0.03), 0.25, 0.45)),
        "pitch_noise_std": float(np.clip(rng.normal(0.20, 0.03), 0.10, 0.35)),
        "power_efficiency": float(np.clip(rng.normal(1.0, 0.02), 0.95, 1.05)),
        "thermal_bias": float(np.clip(rng.normal(0.0, 0.8), -2.0, 2.0)),
        "sensor_noise_scale": float(np.clip(rng.normal(1.0, 0.08), 0.85, 1.15)),
    }
    if overrides:
        params.update(overrides)
    return params



In [28]:
def simulate_site_conditions(start="2025-01-01", n_days=365, freq="10min", seed=random_seed):
    rng = np.random.default_rng(seed)
    time = pd.date_range(start=start, periods=n_days * 24 * 6, freq=freq)
    n = len(time)
    idx = np.arange(n)

    hour_of_day = time.hour + time.minute / 60.0
    day_of_year = time.dayofyear

    temp_noise = np.zeros(n)
    ws_noise = np.zeros(n)

    for t in range(1, n):
        temp_noise[t] = 0.97 * temp_noise[t - 1] + rng.normal(0, 0.25)
        ws_noise[t] = 0.94 * ws_noise[t - 1] + rng.normal(0, 0.45)

    ambient_temp_site = (
        11.0
        + 8.0 * np.sin(2 * np.pi * (day_of_year / 365.25 - 0.15))
        + 4.0 * np.sin(2 * np.pi * (hour_of_day / 24.0 - 0.20))
        + temp_noise
    )

    pressure_pa = (
        101325.0
        + 1200.0 * np.sin(2 * np.pi * idx / (6 * 24 * 8))
        + rng.normal(0, 250, n)
    )

    air_density_site = pressure_pa / (287.05 * (ambient_temp_site + 273.15))

    wind_speed_site = (
        7.5
        + 1.2 * np.sin(2 * np.pi * idx / (6 * 24 * 5))
        + 0.5 * np.sin(2 * np.pi * (hour_of_day / 24.0 + 0.10))
        + ws_noise
    )

    storm_boost = np.zeros(n)
    storm_centers = rng.integers(0, n, size=max(6, n_days // 45))
    for center in storm_centers:
        width = rng.integers(10, 36)
        amplitude = rng.uniform(8.0, 18.0)
        storm_boost += amplitude * np.exp(-0.5 * ((idx - center) / width) ** 2)

    wind_speed_site = np.clip(wind_speed_site + storm_boost, 0.0, 30.0)

    wind_direction_site = np.zeros(n)
    wind_direction_site[0] = 220.0
    mean_direction = wrap_deg(220.0 + 35.0 * np.sin(2 * np.pi * idx / (6 * 24 * 14)))

    for t in range(1, n):
        wind_direction_site[t] = wrap_deg(
            wind_direction_site[t - 1]
            + 0.10 * angle_diff(mean_direction[t], wind_direction_site[t - 1])
            + rng.normal(0, 4.5)
        )

    return pd.DataFrame({
        "time": time,
        "ambient_temp_site": ambient_temp_site,
        "air_density_site": air_density_site,
        "wind_speed_site": wind_speed_site,
        "wind_direction_site": wind_direction_site,
    })



In [29]:
def generate_healthy_device(site_df, device_id, device_params, seed=random_seed):
    rng = np.random.default_rng(seed)
    n = len(site_df)
    noise_scale = device_params["sensor_noise_scale"]

    wind_speed = np.clip(
        site_df["wind_speed_site"].to_numpy() + rng.normal(0, 0.18 * noise_scale, n),
        0.0,
        30.0
    )
    wind_direction = wrap_deg(
        site_df["wind_direction_site"].to_numpy() + rng.normal(0, 1.5 * noise_scale, n)
    )
    ambient_temp = site_df["ambient_temp_site"].to_numpy() + rng.normal(0, 0.25 * noise_scale, n)
    air_density = np.clip(
        site_df["air_density_site"].to_numpy() + rng.normal(0, 0.004 * noise_scale, n),
        1.10,
        1.35
    )

    nacelle_direction = np.zeros(n)
    nacelle_direction[0] = wrap_deg(wind_direction[0] + rng.normal(0, 1.5 * noise_scale))
    for t in range(1, n):
        target_heading = wind_direction[t]
        nacelle_direction[t] = wrap_deg(
            nacelle_direction[t - 1]
            + device_params["yaw_response_gain"] * angle_diff(target_heading, nacelle_direction[t - 1])
            + rng.normal(0, 1.0 * noise_scale)
        )

    nacelle_position = wrap_deg(nacelle_direction + rng.normal(0, 0.7 * noise_scale, n))
    yaw_error = np.abs(angle_diff(wind_direction, nacelle_direction))

    pitch_base = commanded_pitch_angle(wind_speed)
    pitch_std = device_params["pitch_noise_std"]
    pitch_1 = np.clip(pitch_base + rng.normal(0, pitch_std, n), 0.0, 90.0)
    pitch_2 = np.clip(pitch_base + rng.normal(0, pitch_std, n), 0.0, 90.0)
    pitch_3 = np.clip(pitch_base + rng.normal(0, pitch_std, n), 0.0, 90.0)

    pitch_spread = np.maximum.reduce([
        np.abs(pitch_1 - pitch_2),
        np.abs(pitch_1 - pitch_3),
        np.abs(pitch_2 - pitch_3),
    ])

    raw_power = power_curve_kw(wind_speed)
    density_factor = np.clip(air_density / 1.225, 0.90, 1.10)
    yaw_efficiency = np.clip(np.cos(np.deg2rad(yaw_error)), 0.0, 1.0) ** 3
    pitch_efficiency = np.clip(1.0 - 0.012 * (pitch_spread ** 1.15), 0.75, 1.0)

    active_power = (
        raw_power
        * density_factor
        * yaw_efficiency
        * pitch_efficiency
        * device_params["power_efficiency"]
    )

    active_power += rng.normal(0, np.maximum(8.0, 0.015 * np.maximum(raw_power, 1.0)), n)
    active_power = np.clip(active_power, 0.0, device_params["rated_power_kw"])

    rotor_speed = np.where(
        wind_speed < 3.0,
        np.clip(0.3 + 0.7 * wind_speed + rng.normal(0, 0.25 * noise_scale, n), 0.0, None),
        np.where(
            wind_speed < 10.0,
            4.0 + 1.7 * (wind_speed - 3.0) + rng.normal(0, 0.35 * noise_scale, n),
            15.8 + 0.08 * (wind_speed - 10.0) + rng.normal(0, 0.25 * noise_scale, n),
        ),
    )

    rotor_speed = rotor_speed * (0.96 + 0.04 * (active_power / np.maximum(raw_power, 1.0)))
    rotor_speed = np.where(
        active_power < 5.0,
        np.clip(rotor_speed - rng.uniform(1.5, 3.0, n), 0.0, None),
        rotor_speed
    )
    rotor_speed = np.clip(rotor_speed, 0.0, 22.0)

    generator_speed = np.clip(
        rotor_speed * device_params["gearbox_ratio"] + rng.normal(0, 18 * noise_scale, n),
        0.0,
        2200.0
    )

    thermal_bias = device_params["thermal_bias"]
    gearbox_oil_temp = ambient_temp + 17.0 + 0.010 * active_power + 0.10 * rotor_speed + thermal_bias + rng.normal(0, 1.2, n)
    generator_temp = ambient_temp + 20.0 + 0.014 * active_power + 0.0008 * generator_speed + thermal_bias + rng.normal(0, 1.5, n)
    bearing_temp = ambient_temp + 12.0 + 0.007 * active_power + 0.06 * rotor_speed + 0.015 * (yaw_error ** 1.7) + thermal_bias + rng.normal(0, 1.0, n)
    converter_temp = ambient_temp + 14.0 + 0.011 * active_power + thermal_bias + rng.normal(0, 1.3, n)

    df = pd.DataFrame({
        "time": site_df["time"].to_numpy(),
        "device": device_id,
        "active_power": active_power,
        "wind_speed": wind_speed,
        "air_density": air_density,
        "wind_direction": wind_direction,
        "nacelle_direction": nacelle_direction,
        "nacelle_position": nacelle_position,
        "ambient_temp": ambient_temp,
        "rotor_speed": rotor_speed,
        "generator_speed": generator_speed,
        "gearbox_oil_temp": gearbox_oil_temp,
        "generator_temp": generator_temp,
        "bearing_temp": bearing_temp,
        "converter_temp": converter_temp,
        "pitch_blade_angle_1": pitch_1,
        "pitch_blade_angle_2": pitch_2,
        "pitch_blade_angle_3": pitch_3,
    })

    df["is_drifted"] = 0
    df["fault_labels"] = "healthy"
    df["fault_severity"] = "none"
    df["drift_start_time"] = pd.NaT

    for col in FAULT_BINARY_COLUMNS:
        df[col] = 0

    return df



In [30]:
def apply_temperature_fault(df, fault_config):
    profile = build_fault_profile(
        n=len(df),
        start_idx=fault_config["start_idx"],
        ramp_steps=fault_config.get("ramp_steps", 1),
        end_idx=fault_config.get("end_idx"),
        shape=fault_config.get("shape", "linear"),
        max_severity=fault_config.get("max_severity", 1.0),
        ramp_down_steps=fault_config.get("ramp_down_steps", 0),
    )

    if np.all(profile == 0):
        return df, profile

    late_derate = np.maximum(profile - 0.70, 0.0)

    df["gearbox_oil_temp"] += 10.0 * profile
    df["generator_temp"] += 16.0 * profile
    df["bearing_temp"] += 8.0 * profile
    df["converter_temp"] += 6.0 * profile
    df["active_power"] *= (1.0 - 0.05 * late_derate)
    df["active_power"] = np.clip(df["active_power"], 0.0, None)

    return df, profile

In [31]:

def apply_pitch_misalignment_fault(df, fault_config):
    profile = build_fault_profile(
        n=len(df),
        start_idx=fault_config["start_idx"],
        ramp_steps=fault_config.get("ramp_steps", 1),
        end_idx=fault_config.get("end_idx"),
        shape=fault_config.get("shape", "linear"),
        max_severity=fault_config.get("max_severity", 10.0),
        ramp_down_steps=fault_config.get("ramp_down_steps", 0),
    )

    if np.all(profile == 0):
        return df, profile

    df["pitch_blade_angle_2"] += profile
    df["pitch_blade_angle_3"] -= 0.4 * profile

    for col in ["pitch_blade_angle_1", "pitch_blade_angle_2", "pitch_blade_angle_3"]:
        df[col] = np.clip(df[col], 0.0, 90.0)

    pitch_spread = np.maximum.reduce([
        np.abs(df["pitch_blade_angle_1"] - df["pitch_blade_angle_2"]),
        np.abs(df["pitch_blade_angle_1"] - df["pitch_blade_angle_3"]),
        np.abs(df["pitch_blade_angle_2"] - df["pitch_blade_angle_3"]),
    ])

    pitch_efficiency = np.clip(1.0 - 0.012 * (pitch_spread ** 1.15), 0.75, 1.0)
    raw_curve_power = power_curve_kw(df["wind_speed"].to_numpy())
    density_factor = np.clip(df["air_density"].to_numpy() / 1.225, 0.90, 1.10)
    yaw_error = np.abs(angle_diff(df["wind_direction"].to_numpy(), df["nacelle_direction"].to_numpy()))
    yaw_efficiency = np.clip(np.cos(np.deg2rad(yaw_error)), 0.0, 1.0) ** 3

    recomputed_power = raw_curve_power * density_factor * yaw_efficiency * pitch_efficiency
    df["active_power"] = np.minimum(df["active_power"].to_numpy(), recomputed_power)
    df["active_power"] = np.clip(df["active_power"], 0.0, None)

    df["bearing_temp"] += 0.12 * pitch_spread
    df["gearbox_oil_temp"] += 0.05 * pitch_spread

    return df, profile

In [32]:
def apply_yaw_misalignment_fault(df, fault_config):
    profile = build_fault_profile(
        n=len(df),
        start_idx=fault_config["start_idx"],
        ramp_steps=fault_config.get("ramp_steps", 1),
        end_idx=fault_config.get("end_idx"),
        shape=fault_config.get("shape", "linear"),
        max_severity=fault_config.get("max_severity", 18.0),
        ramp_down_steps=fault_config.get("ramp_down_steps", 0),
    )

    if np.all(profile == 0):
        return df, profile

    df["nacelle_direction"] = wrap_deg(df["nacelle_direction"].to_numpy() + profile)
    df["nacelle_position"] = wrap_deg(df["nacelle_position"].to_numpy() + profile)

    yaw_error = np.abs(angle_diff(df["wind_direction"].to_numpy(), df["nacelle_direction"].to_numpy()))
    yaw_efficiency = np.clip(np.cos(np.deg2rad(yaw_error)), 0.0, 1.0) ** 3

    pitch_spread = np.maximum.reduce([
        np.abs(df["pitch_blade_angle_1"] - df["pitch_blade_angle_2"]),
        np.abs(df["pitch_blade_angle_1"] - df["pitch_blade_angle_3"]),
        np.abs(df["pitch_blade_angle_2"] - df["pitch_blade_angle_3"]),
    ])
    pitch_efficiency = np.clip(1.0 - 0.012 * (pitch_spread ** 1.15), 0.75, 1.0)

    raw_curve_power = power_curve_kw(df["wind_speed"].to_numpy())
    density_factor = np.clip(df["air_density"].to_numpy() / 1.225, 0.90, 1.10)

    recomputed_power = raw_curve_power * density_factor * yaw_efficiency * pitch_efficiency
    df["active_power"] = np.minimum(df["active_power"].to_numpy(), recomputed_power)
    df["active_power"] = np.clip(df["active_power"], 0.0, None)

    df["bearing_temp"] += 0.015 * (yaw_error ** 1.7)

    return df, profile

In [33]:
FAULT_REGISTRY = {
    "temperature": apply_temperature_fault,
    "pitch_misalignment": apply_pitch_misalignment_fault,
    "yaw_misalignment": apply_yaw_misalignment_fault,
}

FAULT_COLUMN_MAP = {
    "temperature": "fault_temperature",
    "pitch_misalignment": "fault_pitch_misalignment",
    "yaw_misalignment": "fault_yaw_misalignment",
}


def apply_fault(df, fault_config):
    fault_type = fault_config["type"]
    if fault_type not in FAULT_REGISTRY:
        raise ValueError(f"Unknown fault type: {fault_type}")
    return FAULT_REGISTRY[fault_type](df, fault_config)


def finalize_fault_labels(df, fault_states, fault_start_times):
    n = len(df)

    if not fault_states:
        df["is_drifted"] = 0
        df["fault_labels"] = "healthy"
        df["fault_severity"] = "none"
        df["drift_start_time"] = pd.NaT
        return df

    profiles = list(fault_states.values())
    max_profile = np.maximum.reduce(profiles)
    is_drifted = (max_profile > 0).astype(int)
    severity = severity_label(max_profile)

    fault_labels = np.full(n, "healthy", dtype=object)

    for fault_type, profile in fault_states.items():
        active = profile > 0
        binary_col = FAULT_COLUMN_MAP[fault_type]
        df[binary_col] = active.astype(int)

        active_idx = np.where(active)[0]
        for i in active_idx:
            if fault_labels[i] == "healthy":
                fault_labels[i] = fault_type
            else:
                fault_labels[i] = f"{fault_labels[i]}|{fault_type}"

    drift_start_time = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")
    if fault_start_times:
        first_start = min(fault_start_times)
        drift_start_time[:] = first_start

    df["is_drifted"] = is_drifted
    df["fault_labels"] = fault_labels
    df["fault_severity"] = severity
    df["drift_start_time"] = drift_start_time

    return df



In [34]:
def round_and_clip(df):
    df["active_power"] = np.round(np.clip(df["active_power"], 0.0, None), 2)
    df["wind_speed"] = np.round(np.clip(df["wind_speed"], 0.0, 30.0), 2)
    df["air_density"] = np.round(np.clip(df["air_density"], 1.10, 1.35), 4)
    df["wind_direction"] = np.round(wrap_deg(df["wind_direction"]), 2)
    df["nacelle_direction"] = np.round(wrap_deg(df["nacelle_direction"]), 2)
    df["nacelle_position"] = np.round(wrap_deg(df["nacelle_position"]), 2)
    df["ambient_temp"] = np.round(df["ambient_temp"], 2)
    df["rotor_speed"] = np.round(np.clip(df["rotor_speed"], 0.0, 22.0), 2)
    df["generator_speed"] = np.round(np.clip(df["generator_speed"], 0.0, 2200.0), 2)
    df["gearbox_oil_temp"] = np.round(np.clip(df["gearbox_oil_temp"], -20.0, 120.0), 2)
    df["generator_temp"] = np.round(np.clip(df["generator_temp"], -20.0, 140.0), 2)
    df["bearing_temp"] = np.round(np.clip(df["bearing_temp"], -20.0, 120.0), 2)
    df["converter_temp"] = np.round(np.clip(df["converter_temp"], -20.0, 120.0), 2)
    df["pitch_blade_angle_1"] = np.round(np.clip(df["pitch_blade_angle_1"], 0.0, 90.0), 2)
    df["pitch_blade_angle_2"] = np.round(np.clip(df["pitch_blade_angle_2"], 0.0, 90.0), 2)
    df["pitch_blade_angle_3"] = np.round(np.clip(df["pitch_blade_angle_3"], 0.0, 90.0), 2)
    return df



In [35]:
def simulate_device(site_df, device_config, seed=random_seed):
    rng = np.random.default_rng(seed)
    device_id = device_config["device_id"]
    device_params = make_device_params(rng, device_config.get("device_params"))

    df = generate_healthy_device(site_df, device_id, device_params, seed=seed)

    fault_states = {}
    fault_start_times = []
    steps_per_day = 24 * 6

    for fault in device_config.get("faults", []):
        fault_cfg = dict(fault)

        if "start_idx" not in fault_cfg:
            fault_cfg["start_idx"] = start_day_to_idx(fault_cfg.get("start_day"), steps_per_day)

        if "ramp_steps" not in fault_cfg:
            ramp_days = fault_cfg.get("ramp_days")
            fault_cfg["ramp_steps"] = int(ramp_days * steps_per_day) if ramp_days is not None else 1

        if "end_idx" not in fault_cfg and fault_cfg.get("end_day") is not None:
            fault_cfg["end_idx"] = start_day_to_idx(fault_cfg["end_day"], steps_per_day)

        if "ramp_down_steps" not in fault_cfg:
            ramp_down_days = fault_cfg.get("ramp_down_days")
            fault_cfg["ramp_down_steps"] = int(ramp_down_days * steps_per_day) if ramp_down_days is not None else 0

        df, profile = apply_fault(df, fault_cfg)
        if fault_cfg["type"] in fault_states:
            fault_states[fault_cfg["type"]] = np.maximum(fault_states[fault_cfg["type"]], profile)
        else:
            fault_states[fault_cfg["type"]] = profile

        if np.any(profile > 0):
            first_idx = int(np.where(profile > 0)[0][0])
            fault_start_times.append(site_df.iloc[first_idx]["time"])

    df = finalize_fault_labels(df, fault_states, fault_start_times)
    df = round_and_clip(df)
    return df[ALL_COLUMNS]

In [36]:
def build_fault_event_table(site_df, device_configs):
    rows = []
    steps_per_day = 24 * 6

    for cfg in device_configs:
        for fault in cfg.get("faults", []):
            start_idx = fault.get("start_idx")
            if start_idx is None:
                start_idx = start_day_to_idx(fault.get("start_day"), steps_per_day)

            end_idx = fault.get("end_idx")
            if end_idx is None and fault.get("end_day") is not None:
                end_idx = start_day_to_idx(fault.get("end_day"), steps_per_day)

            start_time = pd.NaT
            end_time = pd.NaT

            if start_idx is not None and 0 <= start_idx < len(site_df):
                start_time = site_df.iloc[start_idx]["time"]

            if end_idx is not None and 0 <= end_idx < len(site_df):
                end_time = site_df.iloc[end_idx]["time"]

            rows.append({
                "device": cfg["device_id"],
                "fault_type": fault["type"],
                "start_time": start_time,
                "end_time": end_time,
                "shape": fault.get("shape", "linear"),
                "max_severity": fault.get("max_severity", 1.0),
                "ramp_steps": fault.get("ramp_steps"),
                "ramp_days": fault.get("ramp_days"),
                "ramp_down_days": fault.get("ramp_down_days"),
            })

    return pd.DataFrame(rows)

In [37]:
def simulate_farm(site_config, device_configs, seed=random_seed):
    site_df = simulate_site_conditions(**site_config, seed=seed)
    all_devices = []

    for i, cfg in enumerate(device_configs):
        device_df = simulate_device(site_df, cfg, seed=seed + i + 1)
        all_devices.append(device_df)

    data = pd.concat(all_devices, ignore_index=True)
    data = data.sort_values(["time", "device"]).reset_index(drop=True)
    events = build_fault_event_table(site_df, device_configs)
    return data, events



In [38]:
site_config = {
    "start": "2025-01-01",
    "n_days": 365,
    "freq": "10min",
}

device_configs = [
    {
        "device_id": "WT001",
        "device_params": {"power_efficiency": 1.00},
        "faults": []
    },
    {
        "device_id": "WT002",
        "device_params": {"power_efficiency": 0.99},
        "faults": [
            {
                "type": "temperature",
                "start_day": 100,
                "ramp_days": 14,
                "shape": "linear",
                "max_severity": 1.0,
            }
        ]
    },
    {
        "device_id": "WT003",
        "device_params": {"power_efficiency": 1.01},
        "faults": [
            {
                "type": "pitch_misalignment",
                "start_day": 180,
                "ramp_steps": 6,
                "shape": "abrupt",
                "max_severity": 10.0,
            }
        ]
    },
    {
        "device_id": "WT004",
        "device_params": {"power_efficiency": 0.98},
        "faults": [
            {
                "type": "yaw_misalignment",
                "start_day": 270,
                "ramp_days": 10,
                "shape": "linear",
                
                "max_severity": 18.0,
            }
        ]
    },
    {
        "device_id": "WT005",
        "device_params": {"power_efficiency": 1.00},
        "faults": [
            {
                "type": "temperature",
                "start_day": 50,
                "ramp_days": 14,
                "end_day": 220,
                "ramp_down_days": 14,
                "shape": "linear",
                "max_severity": 1.0,
            }
        ]
    },
    {
        "device_id": "WT006",
        "device_params": {"power_efficiency": 0.99},
        "faults": [
            {
                "type": "pitch_misalignment",
                "start_day": 120,
                "end_day": 210,
                "shape": "abrupt",
                "max_severity": 10.0,
            }
        ]
    },
    {
        "device_id": "WT007",
        "device_params": {"power_efficiency": 1.01},
        "faults": [
            {
                "type": "yaw_misalignment",
                "start_day": 80,
                "ramp_days": 10,
                "end_day": 260,
                "ramp_down_days": 10,
                "shape": "linear",
                "max_severity": 18.0,
            }
        ]
    },
    {
        "device_id": "WT008",
        "device_params": {"power_efficiency": 0.98},
        "faults": [
            {
                "type": "temperature",
                "start_day": 60,
                "ramp_days": 14,
                "end_day": 160,
                "shape": "linear",
                "max_severity": 1.0,
            }
        ]
    },
    {
        "device_id": "WT009",
        "device_params": {"power_efficiency": 1.00},
        "faults": [
            {
                "type": "yaw_misalignment",
                "start_day": 90,
                "ramp_days": 10,
                "end_day": 230,
                "shape": "linear",
                "max_severity": 18.0,
            }
        ]
    },
    {
        "device_id": "WT010",
        "device_params": {"power_efficiency": 0.99},
        "faults": [
            {
                "type": "temperature",
                "start_day": 60,
                "ramp_days": 14,
                "shape": "linear",
                "max_severity": 1.0,
            },
            {
                "type": "yaw_misalignment",
                "start_day": 200,
                "ramp_days": 10,
                "shape": "linear",
                "max_severity": 18.0,
            }
        ]
    },
]

In [39]:
df, events = simulate_farm(site_config, device_configs, seed=random_seed)

output_dir = Path("../generated_data/time_series/wind_data_with_repair")
output_dir.mkdir(parents=True, exist_ok=True)

df.to_parquet(output_dir / "wind_turbine_stream_data_with_repair.parquet", index=False)
events.to_parquet(output_dir / "wind_turbine_fault_events_with_repair.parquet", index=False)

In [40]:
df.groupby("device")["is_drifted"].mean().round(3)

device
WT001    0.000
WT002    0.726
WT003    0.507
WT004    0.260
WT005    0.466
WT006    0.247
WT007    0.493
WT008    0.274
WT009    0.384
WT010    0.836
Name: is_drifted, dtype: float64

In [41]:
df["fault_labels"].value_counts().head(20)

fault_labels
healthy                         305287
temperature                      97197
yaw_misalignment                 59757
pitch_misalignment               39600
temperature|yaw_misalignment     23759
Name: count, dtype: int64

In [42]:
events

,device,fault_type,start_time,end_time,shape,max_severity,ramp_steps,ramp_days,ramp_down_days
0,WT002,temperature,2025-04-11,NaT,linear,1.0,NaN,14.0,NaN
1,WT003,pitch_misalignment,2025-06-30,NaT,abrupt,10.0,6.0,NaN,NaN
2,WT004,yaw_misalignment,2025-09-28,NaT,linear,18.0,NaN,10.0,NaN
3,WT005,temperature,2025-02-20,2025-08-09,linear,1.0,NaN,14.0,14.0
4,WT006,pitch_misalignment,2025-05-01,2025-07-30,abrupt,10.0,NaN,NaN,NaN
5,WT007,yaw_misalignment,2025-03-22,2025-09-18,linear,18.0,NaN,10.0,10.0
6,WT008,temperature,2025-03-02,2025-06-10,linear,1.0,NaN,14.0,NaN
7,WT009,yaw_misalignment,2025-04-01,2025-08-19,linear,18.0,NaN,10.0,NaN
8,WT010,temperature,2025-03-02,NaT,linear,1.0,NaN,14.0,NaN
9,WT010,yaw_misalignment,2025-07-20,NaT,linear,18.0,NaN,10.0,NaN
